# 체형·목적 기반 AI 코디 추천

입력 검증 → 체형 분석 → 의류 분리 → DeepFashion 라벨 체계 기반 착장 분석 → 코디 추천 → 결과 저장 순서로 실행합니다.

## 0. 팀원별 로컬 경로 설정 — 먼저 이 셀만 수정하세요

절대경로와 프로젝트 폴더 기준 상대경로를 모두 사용할 수 있습니다. 빈 데이터·출력 경로는 프로젝트 내부 기본 폴더를 사용합니다.

In [ ]:
from pathlib import Path
import json
import os
import sys

from IPython.display import display
from PIL import Image

# ======================================================================
# [팀원별 로컬 설정] 이 값들만 본인의 환경에 맞게 수정하세요.
PROJECT_DIR_INPUT = r''                 # 비워두면 현재 폴더에서 자동 탐색
IMAGE_PATH_INPUT = r'data/input_person.jpg'  # 분석할 전신사진
DATA_DIR_INPUT = r''                    # 비워두면 PROJECT_DIR/data
OUTPUT_DIR_INPUT = r''                  # 비워두면 PROJECT_DIR/outputs
FONT_PATH_INPUT = r''                   # 선택: 한글 TTF/TTC 글꼴
RULES_PATH_INPUT = r'FASHION_RULES_MASTER.md'
ATTRIBUTE_HEADS_PATH_INPUT = r'models/fashion_attribute_heads.pt'
# ======================================================================

def resolve_local_path(value, default, base_dir):
    selected = Path(value).expanduser() if value else Path(default)
    if not selected.is_absolute():
        selected = Path(base_dir) / selected
    return selected.resolve()

if PROJECT_DIR_INPUT:
    PROJECT_DIR = Path(PROJECT_DIR_INPUT).expanduser().resolve()
else:
    candidates = [Path.cwd(), Path.cwd() / 'ai_fashion_recommender']
    PROJECT_DIR = next((p.resolve() for p in candidates if (p / 'config.py').exists()), None)
if PROJECT_DIR is None or not (PROJECT_DIR / 'config.py').is_file():
    raise FileNotFoundError('PROJECT_DIR_INPUT에 올바른 프로젝트 폴더를 입력하세요.')

IMAGE_PATH = resolve_local_path(IMAGE_PATH_INPUT, 'data/input_person.jpg', PROJECT_DIR)
DATA_DIR = resolve_local_path(DATA_DIR_INPUT, 'data', PROJECT_DIR)
OUTPUT_DIR = resolve_local_path(OUTPUT_DIR_INPUT, 'outputs', PROJECT_DIR)
RULES_PATH = resolve_local_path(RULES_PATH_INPUT, 'FASHION_RULES_MASTER.md', PROJECT_DIR)
ATTRIBUTE_HEADS_PATH = resolve_local_path(
    ATTRIBUTE_HEADS_PATH_INPUT, 'models/fashion_attribute_heads.pt', PROJECT_DIR
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
if FONT_PATH_INPUT:
    os.environ['FASHION_FONT_PATH'] = str(resolve_local_path(FONT_PATH_INPUT, FONT_PATH_INPUT, PROJECT_DIR))
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from clothing_parser import ClothingParser
from fashion_model import FashionClassifier
from feedback_store import FeedbackStore
from outfit_analyzer import OutfitAnalyzer
from pose_analyzer import PoseAnalyzer
from product_catalog import ProductCatalog
from quality_checker import QualityChecker
from recommendation_engine import RecommendationEngine
from schemas import UserProfile
from virtual_tryon import VirtualTryOnAdapter

print('프로젝트 폴더:', PROJECT_DIR)
print('입력 이미지:', IMAGE_PATH)
print('데이터 폴더:', DATA_DIR)
print('출력 폴더:', OUTPUT_DIR)
print('패션 규칙:', RULES_PATH)
print('학습된 속성 헤드:', ATTRIBUTE_HEADS_PATH if ATTRIBUTE_HEADS_PATH.is_file() else '없음 (zero-shot 대체)')

## 1. 사용자 조건과 모델 설정

In [ ]:
profile = UserProfile(
    purpose='데일리',
    desired_style='캐주얼',
    budget=160_000,
    change_scope='전체 변경',  # 현재 유지 / 상의만 변경 / 하의만 변경 / 전체 변경
    height_cm=None,
    weight_kg=None,
    season='사계절',
)
USE_FASHN_PARSER = True
USE_FASHION_SIGLIP = True
USE_VTON = False

if not IMAGE_PATH.is_file():
    raise FileNotFoundError(
        f'입력 이미지가 없습니다: {IMAGE_PATH}\n'
        '0번 셀의 IMAGE_PATH_INPUT을 본인의 전신사진 경로로 변경하세요.'
    )
if not (DATA_DIR / 'products.csv').is_file():
    raise FileNotFoundError(f'상품 파일이 없습니다: {DATA_DIR / "products.csv"}')
if not RULES_PATH.is_file():
    raise FileNotFoundError(f'규칙 파일이 없습니다: {RULES_PATH}')
display(Image.open(IMAGE_PATH).convert('RGB'))
print(profile.to_dict())

## 2. 입력 사진 품질 검사

In [ ]:
# 이 셀을 다시 실행할 때 이전 MediaPipe 인스턴스를 먼저 정리합니다.
if 'pose_analyzer' in globals():
    pose_analyzer.close()
pose_analyzer = PoseAnalyzer(model_complexity=1)
quality_checker = QualityChecker(pose_analyzer)
# 포즈는 여기서 한 번만 계산하고 이후 단계에서 재사용합니다.
pose_result = pose_analyzer.analyze(IMAGE_PATH)
input_quality = quality_checker.check_input(IMAGE_PATH, pose=pose_result)
print(json.dumps(input_quality, ensure_ascii=False, indent=2))
if not input_quality['passed']:
    raise ValueError('전신사진 품질 기준을 통과하지 못했습니다. issues 항목을 확인하세요.')

## 3. MediaPipe 체형·자세 분석

In [ ]:
if not pose_result.valid:
    raise ValueError('유효한 전신 포즈를 찾지 못했습니다.')
display(pose_analyzer.draw_landmarks(IMAGE_PATH, analysis=pose_result))
print(json.dumps({k: v for k, v in pose_result.to_dict().items() if k != 'landmarks'}, ensure_ascii=False, indent=2))

## 4. 의류 분리와 현재 착장 분석

FASHN으로 픽셀 마스크를 만들고 MediaPipe로 기장을 계산합니다. FashionSigLIP 후보군은 DeepFashion-MultiModal의 소재·패턴·네크라인 라벨에 맞춰 구성했습니다.

In [ ]:
clothing_parser = ClothingParser(use_fashn=USE_FASHN_PARSER)
# 학습된 의류 속성 헤드(Fashionpedia+Fashion200K 학습)가 있으면 FashionSigLIP 특징 위에
# 얹어 옷 종류·소매·기장·핏 등의 인식에 사용하고, 없으면 zero-shot 분류만 사용한다.
fashion_classifier = FashionClassifier(
    enabled=USE_FASHION_SIGLIP,
    attribute_checkpoint=ATTRIBUTE_HEADS_PATH if USE_FASHION_SIGLIP and ATTRIBUTE_HEADS_PATH.is_file() else None,
)
print('FashionSigLIP 실행 장치:', fashion_classifier.device)
print('학습된 속성 헤드:', '사용' if fashion_classifier.trained_attributes_enabled else '없음 (zero-shot만 사용)')
outfit_analyzer = OutfitAnalyzer(clothing_parser, fashion_classifier)
outfit_result, parsed = outfit_analyzer.analyze(IMAGE_PATH, pose_result)
if USE_FASHN_PARSER and parsed['backend'] != 'fashn-human-parser':
    raise RuntimeError('FASHN 파서가 요청되었지만 실제 모델이 사용되지 않았습니다.')
segmentation_preview = clothing_parser.colorize(parsed['segmentation'])
segmentation_path = OUTPUT_DIR / 'fashn_segmentation.jpg'
segmentation_preview.save(segmentation_path)
display(segmentation_preview)
print(json.dumps(outfit_result.to_dict(), ensure_ascii=False, indent=2))
print('의류 분할 결과:', segmentation_path)

## 5. 코디 후보 탐색과 순위 결정

In [ ]:
catalog = ProductCatalog(DATA_DIR / 'products.csv')
# 추천 엔진은 FASHION_RULES_MASTER.md의 R-* 규칙을 직접 읽는다.
recommender = RecommendationEngine(RULES_PATH, catalog)
print('활성 규칙:', len(recommender.active_rule_ids), '/ 문서화된 규칙:', len(recommender.documented_rule_ids))
if recommender.unsupported_rule_ids:
    print('미지원 규칙:', ', '.join(recommender.unsupported_rule_ids))
recommendations = recommender.recommend(profile, pose_result, outfit_result, top_k=3)
for recommendation in recommendations:
    print(f'\n#{recommendation.rank} 총점: {recommendation.total_score:.1f}')
    for product in recommendation.products:
        print(f'  - {product.name} / {product.color} / {product.price:,}원')
    for reason in recommendation.reasons:
        print('  ·', reason)
    for tip in recommendation.styling_tips:
        print('  💡', tip)

## 6. 결과 이미지 생성

In [ ]:
# USE_VTON=True면 CatVTON 디퓨전으로 실제 착장 합성을, False면 추천 보드를 만든다.
# 합성에는 4단계에서 만든 FASHN 마스크(경계)와 세그멘테이션(얼굴·손 보호)을 그대로 사용한다.
if USE_VTON:
    from catvton_tryon import CatVTONTryOn

    tryon = CatVTONTryOn()
else:
    tryon = VirtualTryOnAdapter(enabled=False)
preview_path = tryon.generate(
    person_image=IMAGE_PATH,
    recommendation=recommendations[0],
    output_path=OUTPUT_DIR / 'recommendation_preview.jpg',
    context={
        'upper_mask': parsed.get('upper_mask'),
        'lower_mask': parsed.get('lower_mask'),
        'upper_style_mask': parsed.get('upper_style_mask'),
        'lower_style_mask': parsed.get('lower_style_mask'),
        'segmentation': parsed.get('segmentation'),
    },
)
display(Image.open(preview_path))
print('결과 저장 위치:', preview_path)

## 7. 선택적 피드백 저장과 최종 요약

In [ ]:
SAVE_EXAMPLE_FEEDBACK = False
feedback_store = FeedbackStore(OUTPUT_DIR / 'feedback.jsonl')
if SAVE_EXAMPLE_FEEDBACK:
    feedback_store.append(recommendation_rank=1, action='마음에 들어요', note='테스트')

summary = {
    'input_quality_passed': input_quality['passed'],
    'body_shape': pose_result.body_shape,
    'parser_backend': outfit_result.parser_backend,
    'garments': {
        'upper_type': outfit_result.upper_type,
        'lower_type': outfit_result.lower_type,
        'sleeve_length': outfit_result.sleeve_length,
        'upper_length': outfit_result.upper_length,
        'bottom_length': outfit_result.bottom_length,
        'fit': outfit_result.fit,
        'neckline': outfit_result.neckline,
        'pattern': outfit_result.pattern,
        'material': outfit_result.material,
    },
    'best_score': recommendations[0].total_score,
    'preview_path': str(preview_path),
}
print(json.dumps(summary, ensure_ascii=False, indent=2))
pose_analyzer.close()